In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
from grape import Graph

### Preprocess and assign IDs to NP-KG + train-test split for embedding experiments

In [2]:
KG_PATH = '/ix/rboyce/sbt12/np-kg/resources/knowledge_graphs/'
DIR_PATH = '../data/'

In [4]:
KG_FILE = 'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE-HASHED_v3.0.0.tsv'

In [5]:
npkg_hashed = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v3.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+KG_FILE,
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [6]:
npkg_hashed.get_number_of_connected_components()

(348, 1, 1089125)

In [7]:
npkg_hashed = npkg_hashed.remove_disconnected_nodes()

In [8]:
npkg_hashed.get_number_of_edges()

7786923

### Split into train, validation and test

In [9]:
train, test = npkg_hashed.connected_holdout(
    train_size = 0.8,
    random_state=42,
    include_all_edge_types=True,
    verbose=True
)

In [10]:
print('Train holdout:')
print('Number of edges: ', train.get_number_of_edges())
print('Number of nodes: ', train.get_number_of_nodes())
print('Edge types: ', train.get_number_of_edge_types())

Train holdout:
Number of edges:  6229539
Number of nodes:  1089125
Edge types:  1343


In [11]:
print('Test holdout:')
print('Number of edges: ', test.get_number_of_edges())
print('Number of nodes: ', test.get_number_of_nodes())
print('Edge types: ', test.get_number_of_edge_types())

Test holdout:
Number of edges:  1557384
Number of nodes:  1089125
Edge types:  1343


In [12]:
train_sub, valid = train.connected_holdout(
    train_size = 0.9,
    random_state=42,
    include_all_edge_types=True,
    verbose=True
)

In [13]:
print('Train holdout:')
print('Number of edges: ', train_sub.get_number_of_edges())
print('Number of nodes: ', train_sub.get_number_of_nodes())
print('Edge types: ', train_sub.get_number_of_edge_types())

Train holdout:
Number of edges:  5606586
Number of nodes:  1089125
Edge types:  1343


In [14]:
print('Validation holdout:')
print('Number of edges: ', valid.get_number_of_edges())
print('Number of nodes: ', valid.get_number_of_nodes())
print('Edge types: ', valid.get_number_of_edge_types())

Validation holdout:
Number of edges:  622953
Number of nodes:  1089125
Edge types:  1343


### Dump nodes and edges with IDs

In [15]:
node_mapping = npkg_hashed.get_nodes_mapping()
with open(DIR_PATH+'node_mapping.pickle', 'wb') as fileo:
    pickle.dump(node_mapping, fileo)

In [16]:
##N=5606586 (70%)
train_sub.dump_edges(
    DIR_PATH+'train.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

In [17]:
##N=622953 (10%)
valid.dump_edges(
    DIR_PATH+'valid.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

In [18]:
##N=1557384 (20%)
test.dump_edges(
    DIR_PATH+'test.csv',
    separator=',',
    header = False,
    sources_column_number = 0,
    destinations_column_number = 2,
    edge_types_column_number = 1,
    numeric_node_ids = False,
    directed = True
)

### Load data and confirm format

In [3]:
train = pd.read_csv(DIR_PATH+'hpo_data/train.csv', header=None)
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5606586 entries, 0 to 5606585
Data columns (total 3 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   0       object
 1   1       object
 2   2       object
dtypes: object(3)
memory usage: 128.3+ MB


In [4]:
train.head()

,0,1,2
0,clo:0035452,rdfs:subClassOf,clo:0000374
1,ensembl:ENST00000256362,rdfs:subClassOf,so:0000673
2,ensembl:ENST00000256362,ro:0002510,ncbigene:55237
3,ensembl:ENST00000256362,rdfs:subClassOf,so:0001217
4,dbsnp:rs1057517106,rdfs:subClassOf,so:0001483


In [6]:
valid = pd.read_csv(DIR_PATH+'valid.tsv', header=None, sep='\t')
valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622953 entries, 0 to 622952
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   0       622953 non-null  object
 1   1       622953 non-null  object
 2   2       622953 non-null  object
dtypes: object(3)
memory usage: 14.3+ MB


In [7]:
test = pd.read_csv(DIR_PATH+'test.tsv', header=None, sep='\t')
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1557384 entries, 0 to 1557383
Data columns (total 3 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   0       1557384 non-null  object
 1   1       1557384 non-null  object
 2   2       1557384 non-null  object
dtypes: object(3)
memory usage: 35.6+ MB


In [ ]:
##split train for HPO and save as train and validation
train_sub2 = train.